# 11_e Seed V2 Download Manifest From V1

Purpose: create `raw_videos/{v2_run_id}/download_manifest_v1.parquet` from previous downloaded videos, usually v1. This reuses media files by path and does not copy video bytes.

Runtime: CPU. Run this after creating/selecting the v2 run profile and before `11_a` / `11_b` / `11_c`.


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


## Seed Reuse Manifest


In [ ]:
from sport_pipeline.video.download_sources import seed_download_manifest_from_previous_runs, summarize_download_manifest

VIDEO_REUSE_SETTINGS = stage_settings(RUN_PROFILE, 'video_reuse')
REUSE_PREVIOUS_DOWNLOADS = bool(VIDEO_REUSE_SETTINGS.get('reuse_previous_downloads', False))
SOURCE_RUN_IDS = [str(item) for item in VIDEO_REUSE_SETTINGS.get('source_full_run_ids', [])]
if not SOURCE_RUN_IDS and RUN_ID.endswith("_v2"):
    SOURCE_RUN_IDS = [RUN_ID[:-3] + "_v1"]

target_manifest = BASE_DIR / 'raw_videos' / RUN_ID / 'download_manifest_v1.parquet'
summary_path = BASE_DIR / 'reports/preflight' / f'reuse_download_manifest_{RUN_ID}.json'

print('RUN_ID =', RUN_ID)
print('reuse_previous_downloads =', REUSE_PREVIOUS_DOWNLOADS)
print('source_run_ids =', SOURCE_RUN_IDS)
print('target_manifest =', target_manifest)
print('existing_target_summary_before =', summarize_download_manifest(target_manifest))

if not REUSE_PREVIOUS_DOWNLOADS:
    print('WARNING: profile video_reuse.reuse_previous_downloads is false. This notebook is mainly for v2 reuse; continuing for explicit inspection.')
if not SOURCE_RUN_IDS:
    raise RuntimeError('No source_full_run_ids configured. Create v2 via 10b_run_isolation_check or set execution.video_reuse.source_full_run_ids.')

summary = seed_download_manifest_from_previous_runs(
    base_dir=BASE_DIR,
    target_run_id=RUN_ID,
    source_run_ids=SOURCE_RUN_IDS,
    output_manifest=target_manifest,
    summary_path=summary_path,
)
print('seed_summary =', summary)
print('summary_path =', summary_path)
if int(summary.get("downloaded") or 0) == 0:
    raise RuntimeError('No reusable downloaded videos were found. Check v1 raw_videos manifest and Drive mount paths.')
print('NEXT: run 11_a/11_b/11_c. They will reuse these rows and only download missing/new v2 rows; then run 11_d to merge while preserving this canonical manifest.')
